# Async Python: A Comprehensive Guide

## Table of Contents
1. **Fundamentals of Async Programming**
2. **Event Loop & Coroutines**
3. **Async/Await Syntax**
4. **Common Patterns & Examples**
5. **Error Handling**
6. **Real-World Applications**

## 1. Fundamentals of Async Programming

### What is Asynchronous Programming?

Asynchronous programming allows your program to perform other tasks while waiting for I/O operations (network calls, file reading, database queries) to complete. This is different from synchronous programming where you must wait for one operation to finish before starting the next.

### Key Concepts:
- **Synchronous**: Tasks execute one after another. Program waits for each operation to complete.
- **Asynchronous**: Multiple operations can be in progress. Program doesn't block while waiting for I/O.
- **Concurrency**: Appears to execute multiple tasks simultaneously (on a single thread).
- **Parallelism**: Tasks execute truly simultaneously (multiple threads/processes).

### Why Use Async?
✓ Better resource utilization
✓ Improved responsiveness in I/O-bound applications
✓ Single-threaded concurrent execution
✓ Cleaner thread management (no manual thread creation)

In [ ]:
# Example: Synchronous vs Asynchronous

import time
import asyncio

# ===== SYNCHRONOUS APPROACH =====
def sync_task(name, duration):
    """Simulates I/O operation"""
    print(f"{name} started")
    time.sleep(duration)
    print(f"{name} finished (took {duration}s)")
    return f"Result from {name}"

def run_sync_example():
    """Sequential execution - total time = sum of all tasks"""
    start = time.time()
    
    sync_task("Task 1", 2)
    sync_task("Task 2", 2)
    sync_task("Task 3", 2)
    
    print(f"Total time (sync): {time.time() - start:.2f}s\n")

print("=" * 50)
print("SYNCHRONOUS EXECUTION")
print("=" * 50)
run_sync_example()

In [ ]:
# ===== ASYNCHRONOUS APPROACH =====

async def async_task(name, duration):
    """Async version - uses await to simulate I/O"""
    print(f"{name} started")
    await asyncio.sleep(duration)
    print(f"{name} finished (took {duration}s)")
    return f"Result from {name}"

async def run_async_example():
    """Concurrent execution - tasks run concurrently"""
    start = time.time()
    
    # Create tasks (not awaiting immediately)
    task1 = asyncio.create_task(async_task("Task 1", 2))
    task2 = asyncio.create_task(async_task("Task 2", 2))
    task3 = asyncio.create_task(async_task("Task 3", 2))
    
    # Wait for all tasks to complete
    results = await asyncio.gather(task1, task2, task3)
    
    print(f"Total time (async): {time.time() - start:.2f}s")
    print(f"Results: {results}\n")

print("=" * 50)
print("ASYNCHRONOUS EXECUTION")
print("=" * 50)
await run_async_example()

## 2. The Event Loop & Coroutines

### What is an Event Loop?

The event loop is the core mechanism of async programming in Python. It:
- Manages all coroutines
- Schedules which coroutine runs next
- Switches control between coroutines when they await
- Runs in a single thread

### Coroutines

A coroutine is a special function that can be paused and resumed:
- Defined with `async def`
- Contains `await` expressions
- Can yield control back to the event loop
- Returns a coroutine object (not the result)

In [ ]:
# Understanding Coroutines and the Event Loop

async def simple_coroutine():
    print("Coroutine started")
    await asyncio.sleep(1)
    print("Coroutine resumed after await")
    return "Done!"

# Calling async function returns a coroutine object (not executed yet)
coro = simple_coroutine()
print(f"Type: {type(coro)}")
print(f"Coroutine object: {coro}")

# Must run with asyncio.run() to execute
result = await coro
print(f"Result: {result}\n")

In [ ]:
# Visualizing the Event Loop

async def worker(worker_id, duration):
    print(f"  Worker {worker_id}: Starting (will take {duration}s)")
    await asyncio.sleep(duration)
    print(f"  Worker {worker_id}: Finished")
    return f"Work by {worker_id}"

async def event_loop_demo():
    print("Event loop starts")
    print("Creating 4 tasks...")
    
    # All tasks start, but await immediately yields control
    tasks = [
        asyncio.create_task(worker(1, 3)),
        asyncio.create_task(worker(2, 1)),
        asyncio.create_task(worker(3, 2)),
        asyncio.create_task(worker(4, 1)),
    ]
    
    print("Tasks created and running concurrently\n")
    
    # Wait for all to complete
    results = await asyncio.gather(*tasks)
    print(f"\nAll done! Results: {results}")

await event_loop_demo()

## 3. Async/Await Syntax

### Key Keywords:
- **`async def`**: Defines a coroutine function
- **`await`**: Pauses execution until an awaitable completes
- **`asyncio.create_task()`**: Schedules a coroutine to run
- **`asyncio.gather()`**: Wait for multiple coroutines
- **`asyncio.run()`**: Runs a coroutine (entry point)

In [ ]:
# Async/Await Patterns

# Pattern 1: Sequential awaits (one after another)
async def sequential_pattern():
    print("Sequential Pattern:")
    result1 = await asyncio.sleep(1) or "First"
    print(f"  After first await")
    result2 = await asyncio.sleep(1) or "Second"
    print(f"  After second await")
    return "Sequential done"

# Pattern 2: Concurrent with gather
async def concurrent_pattern():
    print("\nConcurrent Pattern (gather):")
    async def task(name, delay):
        await asyncio.sleep(delay)
        return f"Result from {name}"
    
    # All run concurrently
    results = await asyncio.gather(
        task("A", 2),
        task("B", 1),
        task("C", 3),
    )
    return results

# Pattern 3: Creating tasks for more control
async def task_creation_pattern():
    print("\nTask Creation Pattern:")
    async def work(name):
        await asyncio.sleep(1)
        return f"Done: {name}"
    
    # Create tasks (start immediately)
    t1 = asyncio.create_task(work("Task 1"))
    t2 = asyncio.create_task(work("Task 2"))
    t3 = asyncio.create_task(work("Task 3"))
    
    # Collect later
    results = await asyncio.gather(t1, t2, t3)
    return results

# Pattern 4: Waiting for first completion
async def first_completion_pattern():
    print("\nFirst Completion Pattern:")
    async def delayed_task(seconds, name):
        await asyncio.sleep(seconds)
        return f"{name} done"
    
    tasks = [
        asyncio.create_task(delayed_task(3, "Slow")),
        asyncio.create_task(delayed_task(1, "Fast")),
    ]
    
    # Returns first completed task
    done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
    first_result = done.pop().result()
    print(f"  First completed: {first_result}")
    
    # Clean up remaining
    for task in pending:
        task.cancel()
    
    return first_result

print("=" * 50)
print("ASYNC/AWAIT PATTERNS")
print("=" * 50)

await sequential_pattern()
print()
results = await concurrent_pattern()
print(f"  Results: {results}")
print()
results = await task_creation_pattern()
print(f"  Results: {results}")
print()
await first_completion_pattern()

## 4. Common Use Cases & Practical Examples

### Common Async Use Cases:
- Network requests (HTTP calls)
- File I/O operations
- Database queries
- WebSocket connections
- Scheduled tasks
- Building servers

In [ ]:
# Example 1: Simulating HTTP requests

async def fetch_url(url, delay):
    """Simulate fetching a URL"""
    print(f"Fetching {url}...")
    await asyncio.sleep(delay)  # Simulates network delay
    return f"Content from {url}"

async def batch_fetch():
    """Fetch multiple URLs concurrently"""
    urls = [
        ("http://api1.com", 1),
        ("http://api2.com", 2),
        ("http://api3.com", 1.5),
    ]
    
    start = time.time()
    
    # Fetch all concurrently
    results = await asyncio.gather(*[
        fetch_url(url, delay) for url, delay in urls
    ])
    
    elapsed = time.time() - start
    print(f"Fetched {len(results)} URLs in {elapsed:.2f}s")
    for result in results:
        print(f"  - {result}")

print("=" * 50)
print("EXAMPLE 1: Batch URL Fetching")
print("=" * 50)
await batch_fetch()
print()

In [ ]:
# Example 2: Producer-Consumer Pattern

async def producer(queue, producer_id, count):
    """Produces items and puts them in queue"""
    for i in range(count):
        item = f"Item-{producer_id}-{i}"
        print(f"Producer {producer_id}: Producing {item}")
        await queue.put(item)
        await asyncio.sleep(0.5)
    print(f"Producer {producer_id}: Done")

async def consumer(queue, consumer_id):
    """Consumes items from queue"""
    while True:
        try:
            item = await asyncio.wait_for(queue.get(), timeout=2)
            print(f"  Consumer {consumer_id}: Processing {item}")
            await asyncio.sleep(0.3)
            queue.task_done()
        except asyncio.TimeoutError:
            print(f"  Consumer {consumer_id}: Queue empty, stopping")
            break

async def producer_consumer_demo():
    queue = asyncio.Queue()
    
    # Create producers and consumers
    producers = [
        asyncio.create_task(producer(queue, 1, 3)),
        asyncio.create_task(producer(queue, 2, 2)),
    ]
    
    consumers = [
        asyncio.create_task(consumer(queue, 1)),
        asyncio.create_task(consumer(queue, 2)),
    ]
    
    # Wait for producers to finish
    await asyncio.gather(*producers)
    
    # Wait for consumers to process all items
    await asyncio.gather(*consumers)

print("=" * 50)
print("EXAMPLE 2: Producer-Consumer Pattern")
print("=" * 50)
await producer_consumer_demo()
print()

In [ ]:
# Example 3: Timeout Handling

async def long_operation(duration, name):
    """Simulates a long-running operation"""
    try:
        print(f"{name}: Starting (duration: {duration}s)")
        await asyncio.sleep(duration)
        print(f"{name}: Completed")
        return f"Success: {name}"
    except asyncio.TimeoutError:
        print(f"{name}: TIMEOUT!")
        return f"Timeout: {name}"

async def timeout_demo():
    """Demonstrates timeout handling"""
    tasks = [
        asyncio.create_task(long_operation(1, "Fast Task")),
        asyncio.create_task(long_operation(5, "Slow Task")),
        asyncio.create_task(long_operation(2, "Medium Task")),
    ]
    
    # Set timeout for each task
    results = []
    for task in tasks:
        try:
            result = await asyncio.wait_for(task, timeout=2)
            results.append(result)
        except asyncio.TimeoutError:
            results.append(f"Task timed out")
    
    print(f"\nResults:")
    for r in results:
        print(f"  - {r}")

print("=" * 50)
print("EXAMPLE 3: Timeout Handling")
print("=" * 50)
await timeout_demo()
print()

## 5. Error Handling in Async Code

In [ ]:
# Error Handling Patterns

async def risky_operation(task_id, should_fail=False):
    """Task that might fail"""
    await asyncio.sleep(0.5)
    if should_fail:
        raise ValueError(f"Error in task {task_id}")
    return f"Success: {task_id}"

# Pattern 1: Try-except in tasks
async def error_handling_pattern1():
    print("Pattern 1: Try-Except")
    
    async def safe_operation(task_id, should_fail):
        try:
            return await risky_operation(task_id, should_fail)
        except ValueError as e:
            print(f"  Caught error: {e}")
            return f"Handled error for {task_id}"
    
    results = await asyncio.gather(
        safe_operation(1, False),
        safe_operation(2, True),
        safe_operation(3, False),
    )
    print(f"  Results: {results}\n")

# Pattern 2: Gather with return_exceptions
async def error_handling_pattern2():
    print("Pattern 2: Gather with return_exceptions=True")
    
    results = await asyncio.gather(
        risky_operation(1, False),
        risky_operation(2, True),
        risky_operation(3, False),
        return_exceptions=True  # Exceptions are returned, not raised
    )
    
    for i, result in enumerate(results):
        if isinstance(result, Exception):
            print(f"  Task {i+1}: Error - {result}")
        else:
            print(f"  Task {i+1}: {result}")
    print()

# Pattern 3: Custom exception handling
async def error_handling_pattern3():
    print("Pattern 3: Custom Handlers")
    
    async def task_with_handler(task_id):
        try:
            await asyncio.sleep(0.3)
            if task_id == 2:
                raise RuntimeError(f"Task {task_id} failed")
            return f"OK: {task_id}"
        except RuntimeError as e:
            # Custom handling
            return f"Recovered from {e}"
    
    tasks = [asyncio.create_task(task_with_handler(i)) for i in range(1, 4)]
    results = await asyncio.gather(*tasks)
    
    for result in results:
        print(f"  {result}")
    print()

print("=" * 50)
print("ERROR HANDLING PATTERNS")
print("=" * 50)

await error_handling_pattern1()
await error_handling_pattern2()
await error_handling_pattern3()

## 6. Advanced Topics & Best Practices

### Async Context Managers
Used with `async with` to manage resources in async code.

### Async Generators
Functions that yield values asynchronously using `async for`.

### Async Decorators
Functions that wrap and enhance async coroutines.

### Best Practices:
1. Never block the event loop with blocking I/O
2. Use `asyncio.create_task()` to schedule work immediately
3. Handle exceptions explicitly with `return_exceptions=True`
4. Use timeouts for operations that might hang
5. Avoid mixing sync and async code
6. Use async context managers for resource cleanup

In [ ]:
# Advanced: Async Context Manager

class AsyncDatabase:
    """Simulates an async database connection"""
    
    def __init__(self, name):
        self.name = name
        self.connected = False
    
    async def __aenter__(self):
        print(f"  Connecting to {self.name}...")
        await asyncio.sleep(0.5)
        self.connected = True
        print(f"  Connected to {self.name}")
        return self
    
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print(f"  Closing {self.name}...")
        await asyncio.sleep(0.3)
        self.connected = False
        print(f"  Closed {self.name}")
        return False  # Don't suppress exceptions
    
    async def query(self, sql):
        if not self.connected:
            raise RuntimeError("Not connected")
        await asyncio.sleep(0.2)
        return f"Query '{sql}' result"

async def async_context_manager_demo():
    print("Using Async Context Manager:")
    async with AsyncDatabase("MainDB") as db:
        result = await db.query("SELECT * FROM users")
        print(f"  {result}")
    print()  # Context manager has exited

print("=" * 50)
print("ADVANCED TOPICS")
print("=" * 50)

await async_context_manager_demo()

In [ ]:
# Advanced: Async Generator

async def async_generator_demo():
    """Demonstrates async generators"""
    
    async def data_source(count):
        """Async generator that yields values"""
        print("  Generator starting")
        for i in range(count):
            await asyncio.sleep(0.3)
            print(f"    Yielding {i}")
            yield f"Data-{i}"
        print("  Generator finished")
    
    print("Async Generator:")
    async for item in data_source(3):
        print(f"  Received: {item}")
    print()

await async_generator_demo()

In [ ]:
# Advanced: Async Decorator

def async_timer(func):
    """Decorator that times async functions"""
    async def wrapper(*args, **kwargs):
        start = time.time()
        print(f"  [{func.__name__}] Starting...")
        result = await func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"  [{func.__name__}] Took {elapsed:.2f}s")
        return result
    return wrapper

@async_timer
async def timed_operation(duration):
    """Operation with timing decorator"""
    await asyncio.sleep(duration)
    return "Done"

print("Async Decorator:")
result = await timed_operation(1)
print(f"  Result: {result}\n")

## 7. Real-World Example: Building an Async Web Scraper

A practical example combining multiple async concepts:
- Multiple concurrent requests
- Timeout handling
- Error handling
- Progress tracking

In [ ]:
# Real-World Example: Async Web Scraper Simulator

class AsyncScraper:
    def __init__(self, max_concurrent=3):
        self.semaphore = asyncio.Semaphore(max_concurrent)
        self.session = None
        self.results = []
    
    async def __aenter__(self):
        print("Opening scraper session")
        self.session = "active"
        return self
    
    async def __aexit__(self, *args):
        print("Closing scraper session")
        self.session = None
    
    async def fetch_page(self, url):
        """Fetch a single page with rate limiting"""
        async with self.semaphore:
            try:
                print(f"  Fetching {url}")
                # Simulate network delay
                delay = 0.5 + (hash(url) % 10) * 0.1
                await asyncio.sleep(delay)
                
                # Simulate occasional failures
                if hash(url) % 7 == 0:
                    raise RuntimeError(f"Failed to fetch {url}")
                
                return {
                    "url": url,
                    "status": "success",
                    "content_length": hash(url) % 10000
                }
            except RuntimeError as e:
                return {
                    "url": url,
                    "status": "error",
                    "error": str(e)
                }
    
    async def scrape_urls(self, urls):
        """Scrape multiple URLs concurrently"""
        print(f"Starting to scrape {len(urls)} URLs\n")
        start = time.time()
        
        # Fetch all concurrently
        results = await asyncio.gather(*[
            asyncio.create_task(self.fetch_page(url)) 
            for url in urls
        ])
        
        elapsed = time.time() - start
        
        # Print summary
        print(f"\nCompleted {len(results)} requests in {elapsed:.2f}s")
        successful = sum(1 for r in results if r["status"] == "success")
        failed = sum(1 for r in results if r["status"] == "error")
        
        print(f"Successful: {successful}")
        print(f"Failed: {failed}")
        
        return results

# Run the scraper
async def scraper_demo():
    urls = [
        "http://example.com/page1",
        "http://example.com/page2",
        "http://example.com/page3",
        "http://example.com/page4",
        "http://example.com/page5",
        "http://example.com/page6",
    ]
    
    async with AsyncScraper(max_concurrent=2) as scraper:
        results = await scraper.scrape_urls(urls)
        
        print("\nDetailed Results:")
        for r in results:
            if r["status"] == "success":
                print(f"  ✓ {r['url']} ({r['content_length']} bytes)")
            else:
                print(f"  ✗ {r['url']} - {r['error']}")

print("=" * 50)
print("REAL-WORLD EXAMPLE: Web Scraper")
print("=" * 50)

await scraper_demo()

## Summary & Quick Reference

### When to Use Async:
✓ I/O-bound operations (network, files, database)
✓ Multiple concurrent operations
✓ Building responsive servers
✗ CPU-bound operations (use multiprocessing instead)

### Common Async Functions:
| Function | Purpose |
|----------|---------|
| `asyncio.run(coro)` | Execute a coroutine (entry point) |
| `await coro` | Wait for a coroutine to complete |
| `asyncio.create_task(coro)` | Schedule a coroutine |
| `asyncio.gather(*coros)` | Wait for multiple coroutines |
| `asyncio.wait(tasks)` | More control over waiting |
| `asyncio.sleep(delay)` | Async sleep |
| `asyncio.Queue` | Thread-safe queue for async code |
| `asyncio.Lock` | Async lock for synchronization |

### Key Takeaways:
1. Async allows concurrent I/O-bound operations in a single thread
2. The event loop manages all coroutines
3. Use `await` to pause and resume at I/O boundaries
4. Combine tasks with `asyncio.gather()` or `asyncio.wait()`
5. Always handle exceptions explicitly
6. Use timeouts to prevent hanging
7. Async context managers provide clean resource management